In [1]:
from langchain_ollama import ChatOllama

# Step 2: Connect using the direct local Ollama channel
Model = ChatOllama(
    model="gemma4:e4b",
    base_url="http://127.0.0.1:11434", # Notice: NO '/v1' path suffix needed here
    temperature=1.0  ,                  # Recommended baseline sampling for Gemma 4
    num_ctx = 16384
)


In [2]:
from langchain_anthropic import ChatAnthropic
from langchain_community.tools.tavily_search import TavilySearchResults
#from langchain_tavily import TavilySearch
from deepagents import create_deep_agent
import os

In [4]:
#!conda install conda-forge::langchain-tavily -v

In [5]:
# 1. Provide Domain-Specific Tools
# The agent will automatically blend these with its native file & planning tools.
web_search_tool = TavilySearchResults(
    max_results=5, 
    description="Useful for finding up-to-date market, competitor, and industry data."
)
custom_tools = [web_search_tool]


C:\Users\maxim\AppData\Local\Temp\ipykernel_17700\1576097492.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(


In [6]:

# 2. Define the Specialized Sub-Agents
# Sub-agents spawn dynamically inside isolated context windows to keep the main chat clean.
market_analyst_subagent = {
    "name": "market_analyst",
    "description": "Specialist in analyzing market size, growth trends, and raw web search data.",
    "system_prompt": """You are an expert Market Analyst. Your job is to take raw data provided by the user, 
    analyze market trends, and write a summary. Write your intermediate analysis directly to files.""",
    "tools": [web_search_tool]
}


In [7]:

critique_subagent = {
    "name": "report_reviewer",
    "description": "Specialist in reviewing final markdown reports for logical consistency and depth.",
    "system_prompt": """You are an elite research critic. Read the generated report from the file system. 
    Point out structural gaps, missing metrics, or weak arguments. Provide actionable edit requests.""",
    "tools": []  # Requires no custom tools; relies on the native file system tools
}

In [8]:


subagents_list = [market_analyst_subagent, critique_subagent]


In [9]:

# 3. Formulate the Orchestration Prompt
# Instruct the main agent on how to coordinate its planning, files, and sub-agents.
system_instruction = """You are an autonomous Deep Market Research Executive. 
Your objective is to compile an exhaustive market intelligence report on requested topics.

EXECUTIVE WORKFLOW METHODOLOGY:
1. PLANNING: Immediately break down the goal using the 'write_todos' tool.
2. DISCOVERY: Delegate data collection to the 'market_analyst' sub-agent using the 'task' tool.
3. CONTEXT MANAGEMENT: Read incoming data and use 'write_file' to create 'market_analysis_raw.md'.
4. DRAFTING: Synthesize the final comprehensive report to a file named 'final_market_report.md'.
5. QUALITY ASSURANCE: Delegate 'final_market_report.md' to the 'report_reviewer' sub-agent.
6. ITERATION: Use 'edit_file' to adjust the report based on critique before finalizing your response.
"""

In [10]:

# 4. Instantiate the Core Deep Agent Harness
# The harness automatically wraps the LangGraph loops, State, and Native Middleware.
print("Initializing deep agent configuration...")
research_agent = create_deep_agent(
    model = Model ,  # High-token context frontier model
    tools = custom_tools,
    system_prompt = system_instruction,
    subagents=subagents_list,
    # Optional: Pause execution before critical file edits for human intervention
    interrupt_on={"edit_file": True} 
)


Initializing deep agent configuration...


In [ ]:


# 5. Execute the Run Loop
# The deep agent can take dozens of turns in the background managing its own state.
query = "Analyze the competitive landscape of consumer humanoid robotics companies in june 2026."

print(f"Starting long-horizon task: '{query}'")
result = research_agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 50}  # Allow deep iteration steps
)

# 6. Extract results from final response loop
print("\n--- Execution Complete ---")
print(result)
print(result["messages"][-1].content)


Starting long-horizon task: 'Analyze the competitive landscape of consumer humanoid robotics companies in june 2026.'
